# Design controls: two-stage vs single-stage, class weights, SMOTE

The earlier comparisons mixed several changes at once. In the first baseline notebook the single-stage LightGBM was trained with `lgb.train`, where the `class_weight` option is not used (it only exists in the scikit-learn wrapper), so that model was effectively **unweighted**. The SMOTE runs also held out part of the *oversampled* data for early stopping, so validation rows could have synthetic twins in training. This notebook separates the three factors.

| Factor | Levels |
|---|---|
| Design | single-stage (10-class) vs two-stage (attack or not, then which attack) |
| Class weights | none vs `balanced` |
| Training data | original rows vs original + synthetic BorderlineSMOTE rows |

**Fair validation for every condition:** the validation slice (15% of the *original* training rows, stratified, seed-dependent) contains real rows only and is identical across conditions for a given seed. Synthetic rows, when used, are added to the training part only. Early stopping uses this slice. The test set is touched only for the final scores.

All other settings are the baseline's (learning rate 0.08, 63 leaves, up to 300 rounds, early stopping 20). 3 seeds per condition. Test predictions are saved for the bootstrap analysis. LightGBM runs on CPU here (the Kaggle build has no GPU support); the GPU is used in the GNN notebooks.

In [ ]:
import time

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

DEBUG = False   # True = quick smoke test on a small random sample
SEEDS = [42] if DEBUG else [42, 43, 44]
DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"

## 1. Load data

`train_smote.parquet` starts with exactly the rows of `train.parquet` (same order) followed by the synthetic rows, which is verified below.

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
smote_df = pd.read_parquet(f"{DATASET_PATH}/train_smote.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")

n_real = len(train_df)
assert np.allclose(smote_df.iloc[:n_real].drop(columns="label").to_numpy(float), train_df.drop(columns="label").to_numpy(float))
assert (smote_df.iloc[:n_real]["label"].to_numpy() == train_df["label"].to_numpy()).all()
synthetic = smote_df.iloc[n_real:]
print("real train rows:", n_real, "| synthetic rows:", len(synthetic), synthetic["label"].value_counts().to_dict())

if DEBUG:
    train_df, test_df = train_df.sample(150_000, random_state=0), test_df.sample(30_000, random_state=0)

class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
normal_label = list(class_names).index("Normal")
attack_ids = [i for i in range(n_classes) if i != normal_label]
remap = {orig: new for new, orig in enumerate(attack_ids)}
features = [c for c in train_df.columns if c != "label"]
y_test = test_df["label"].to_numpy()
print("train", len(train_df), "| test", len(test_df), "| features", len(features))

## 2. Models

`weighted=True` uses `class_weight="balanced"`; `False` leaves the loss unweighted. The two-stage model has the same stages as the baseline.

In [ ]:
STOP = [lgb.early_stopping(20), lgb.log_evaluation(0)]


def params(seed, weighted):
    return dict(class_weight="balanced" if weighted else None, learning_rate=0.08, num_leaves=63,
                n_estimators=300, random_state=seed, verbose=-1)


def single_stage(X_tr, y_tr, X_val, y_val, X_test, seed, weighted):
    model = lgb.LGBMClassifier(objective="multiclass", num_class=n_classes, **params(seed, weighted))
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=STOP)
    return model.predict(X_test)


def two_stage(X_tr, y_tr, X_val, y_val, X_test, seed, weighted):
    stage1 = lgb.LGBMClassifier(objective="binary", **params(seed, weighted))
    stage1.fit(X_tr, (y_tr != normal_label).astype(int),
               eval_set=[(X_val, (y_val != normal_label).astype(int))], callbacks=STOP)
    a_tr, a_val = (y_tr != normal_label).to_numpy(), (y_val != normal_label).to_numpy()
    stage2 = lgb.LGBMClassifier(objective="multiclass", num_class=len(attack_ids), **params(seed, weighted))
    stage2.fit(X_tr[a_tr], y_tr[a_tr].map(remap), eval_set=[(X_val[a_val], y_val[a_val].map(remap))], callbacks=STOP)
    pred = np.full(len(X_test), normal_label)
    flagged = np.where(stage1.predict(X_test) == 1)[0]
    pred[flagged] = np.array(attack_ids)[stage2.predict(X_test.iloc[flagged])]
    return pred


# name: (model function, weighted, add synthetic rows to the training part)
CONDITIONS = {
    "single-stage, unweighted, original": (single_stage, False, False),
    "single-stage, weighted, original": (single_stage, True, False),
    "two-stage, weighted, original": (two_stage, True, False),
    "single-stage, weighted, + SMOTE rows": (single_stage, True, True),
    "two-stage, weighted, + SMOTE rows": (two_stage, True, True),
}

## 3. Run all conditions

In [ ]:
rows, preds = [], {}
for seed in SEEDS:
    tr, val = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=seed)   # real rows only
    for name, (fn, weighted, add_synth) in CONDITIONS.items():
        t0 = time.time()
        fit_df = pd.concat([tr, synthetic]) if add_synth else tr
        pred = fn(fit_df[features], fit_df["label"], val[features], val["label"], test_df[features], seed, weighted)
        f1s = f1_score(y_test, pred, labels=np.arange(n_classes), average=None, zero_division=0)
        preds[f"{name}__{seed}"] = pred.astype("int8")
        rows.append({"condition": name, "seed": seed, "test macro F1": f1s.mean(),
                     **{f"F1 {c}": f for c, f in zip(class_names, f1s)}})
        print(f"{name:<40} seed {seed} | test macro F1 {f1s.mean():.4f} | {time.time() - t0:.0f}s", flush=True)

results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/design_controls_runs.csv", index=False)
pd.DataFrame(preds, index=test_df.index).to_parquet("/kaggle/working/design_controls_test_pred.parquet")

## 4. Results (mean and std over seeds)

In [ ]:
summary = results.groupby("condition", sort=False)["test macro F1"].agg(["mean", "std"])
print(summary.round(4).to_string())

p = results.pivot(index="seed", columns="condition", values="test macro F1")
pairs = [("two-stage, weighted, original", "single-stage, weighted, original", "design: two-stage vs single-stage (both weighted, original data)"),
         ("single-stage, weighted, original", "single-stage, unweighted, original", "class weights (single-stage, original data)"),
         ("two-stage, weighted, original", "two-stage, weighted, + SMOTE rows", "SMOTE rows removed (two-stage)"),
         ("single-stage, weighted, original", "single-stage, weighted, + SMOTE rows", "SMOTE rows removed (single-stage)")]
for a, b, label in pairs:
    d = p[a] - p[b]
    print(f"{label}: per seed {d.round(4).tolist()} | mean {d.mean():+.4f}")

import matplotlib.pyplot as plt
import seaborn as sns

order = list(CONDITIONS)
plt.figure(figsize=(10, 4))
sns.barplot(data=results, y="condition", x="test macro F1", order=order, errorbar="sd", capsize=0.15, color="steelblue", alpha=0.8)
sns.stripplot(data=results, y="condition", x="test macro F1", order=order, color="black", size=5)
plt.xlim(0.5, 0.7)
plt.title("Design, class weights and SMOTE rows: test macro F1 (mean and std over seeds)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()